# Legislative Content Extraction (ADK) - Evaluation Smoke Test

Fetch the dataset from Langfuse and run the agent on the first item to verify the pipeline works end-to-end.

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# Ensure cwd is the repo root so relative paths work
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
# Walk up until we find pyproject.toml
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

load_dotenv(verbose=True)
print(f"Working directory: {Path.cwd()}")

Working directory: /home/coder/eval-agents


## 1. Fetch dataset from Langfuse

In [2]:
from aieng.agent_evals.async_client_manager import AsyncClientManager

client_manager = AsyncClientManager.get_instance()
langfuse_client = client_manager.langfuse_client

DATASET_NAME = "legislative-docs-MN"
FILES_DIR = Path("implementations/legislative_content_extraction/files").resolve()
MAX_CONCURRENCY = 3

dataset = langfuse_client.get_dataset(DATASET_NAME)
print(f"Dataset name: {dataset.name}")
print(f"Items:   {len(dataset.items)}")
print(f"Files dir:     {FILES_DIR}")
print(f"Concurrency:   {MAX_CONCURRENCY}")

Dataset name: legislative-docs-MN
Items:   5
Files dir:     /home/coder/eval-agents/implementations/legislative_content_extraction/files
Concurrency:   3


## 2. Inspect the first item

In [4]:
first_item = dataset.items[0]

print("Input:")
print(json.dumps(first_item.input, indent=2))
print("\nExpected output:")
print(json.dumps(first_item.expected_output, indent=2))
print("\nMetadata:")
print(json.dumps(first_item.metadata, indent=2))

Input:
{
  "prompt": "Extract legislative metadata from this bill.",
  "record_id": "MN_SR0010",
  "pdf_file_name": "MN_SR0010.pdf",
  "html_page_link": "https://www.senate.mn/resolutions/display_resolution.html?ls=94&bill_type=SR&bill_number=0010&ss_number=0&ss_year=2025"
}

Expected output:
{
  "title": "A Senate resolution relating to standing committees; setting the schedule of standing committee meetings.",
  "summary": "Establishes the membership and meeting schedule (times and locations) for the standing committees of the Minnesota Senate for the 94th legislative session.",
  "sponsors": [],
  "chamber_code": "SENATE",
  "session_code": "MN_2025_2026_R1",
  "measure_number": "10",
  "jurisdiction_code": "MN",
  "measure_type_code": "SENATE_RESOLUTION",
  "sections_affected": []
}

Metadata:
{
  "id": "MN_SR0010"
}


## 3. Run evaluation on the dataset

In [5]:
from aieng.agent_evals.legislative_content_extraction.evaluation.offline import evaluate

await evaluate(
    dataset_name=DATASET_NAME,
    files_dir=FILES_DIR,
    max_concurrency=MAX_CONCURRENCY,
)

2026-03-25 16:35:44,943 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-03-25 16:35:44,984 INFO aieng.agent_evals.langfuse: Langfuse tracing initialized successfully (endpoint: https://us.cloud.langfuse.com/api/public/otel)
2026-03-25 16:35:44,986 WARNING google_genai._api_client: Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-03-25 16:35:45,381 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metadata from: /home/coder/eval-agents/implementations/legislative_content_extraction/files/MN_SR0010/MN_SR0010.pdf
2026-03-25 16:35:45,388 WARNING google_genai._api_client: Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-03-25 16:35:45,507 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-25 16:35:45,513 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metada

Evaluating traces ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 0:00:00 0:03:08m 0:03:08


2026-03-25 16:39:17,170 INFO aieng.agent_evals.evaluation.trace: Trace 173674878c8d4a5ab63ea8b0a17a9fc7 evaluation skipped: Trace did not become ready within wait window.
2026-03-25 16:39:17,171 INFO aieng.agent_evals.evaluation.trace: Trace acf7fced7d4e76b5ed31d884b2a14d7f evaluation skipped: Trace did not become ready within wait window.


AttributeError: 'EvaluationResult' object has no attribute 'item_results'